In [1]:
#@title Import Dependencies
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.datasets as dsets
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch.utils.data import DataLoader, TensorDataset
 
# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
#@title Function to plot an image with the scale color bar
def plotWithColor(data, title=''):
  if torch.is_tensor(data):
    data=data.cpu().numpy()
  ax = plt.subplot()
  im = ax.imshow(data)
  divider = make_axes_locatable(ax)
  cax = divider.append_axes("right", size="5%", pad=0.05)

  plt.colorbar(im, cax=cax)

  if len(title)>0:
    plt.title(title)

  plt.show()

In [3]:
#@title Downloading MNIST data

train_data = dsets.MNIST(root = './data', train = True,
                        transform = transforms.ToTensor(), download = True)

test_data = dsets.MNIST(root = './data', train = False,
                       transform = transforms.ToTensor())

In [4]:
#@title Loading the data

batch_size = 100 # the size of input data took for one iteration

train_gen = torch.utils.data.DataLoader(dataset = train_data,
                                             batch_size = batch_size,
                                             shuffle = True)

test_gen = torch.utils.data.DataLoader(dataset = test_data,
                                      batch_size = batch_size,
                                      shuffle = False)

In [5]:
#@title Define Hyperparameters
input_size = 2*784 # img_size = (28,28) ---> 28*28=784 in total x 2 for side by side images
num_classes = 10 # number of output classes discrete range [0,9]
num_epochs = 5 # 20 # number of times which the entire dataset is passed throughout the model
lr = 1e-2 # size of step

In [6]:
#@title Define Dendrite Hyperparameters

excitatory_synapses_per_dendrite_branch=50
inhibitory_synapses_per_dendrite_branch=1

dendrite_layers_n = 2
dendrite_branch_per_dendrite = [3, 3] # list with one fewer entries than dendrite_layers_n
dendrite_per_soma=10


# dendrite_branch_per_soma=dendrite_branch_per_dendrite*dendrite_per_soma

e_to_i_ratio=excitatory_synapses_per_dendrite_branch/inhibitory_synapses_per_dendrite_branch

soma_n=num_classes
synapses_per_inhibitory_unit=200
inhibitory_units_n=1

sigmoid_offset=5

# dendrite_n=soma_n*dendrite_per_soma
# dendrite_branch_n=dendrite_n*dendrite_branch_per_dendrite

dendrite_branch_n = [None] * dendrite_layers_n
dendrite_branch_n[0] = dendrite_per_soma * soma_n

for i in range(1, dendrite_layers_n):
    dendrite_branch_n[i] = dendrite_branch_n[i-1] * dendrite_branch_per_dendrite[i-1]

shunting=False

In [7]:
sum(dendrite_branch_n)

400

In [8]:
#@title Index arrays
index_list1=torch.tensor(range(batch_size))
index_list2=torch.tensor(list(range(1,batch_size))+[0])
modulo_base=num_classes

def sum_labels(labels_in):
  return torch.remainder(labels_in[index_list1]+labels_in[index_list2], modulo_base)

def transform_images(images_in):
  return torch.cat((images_in[index_list1, :], images_in[index_list2, :]), 1)


In [9]:
#@title Define layer class that only keeps K top weights
class TopKLinear(nn.Linear):
  """ Custom Linear layer that only uses the stongest K weights """
  def __init__(self, in_features, out_features, keep_in_features, bias=False):
    super().__init__(in_features, out_features, bias=bias)
    self.keep_in_features = keep_in_features
    self.topk_indices=[]
    self.mask=0*self.weight.data
    self.net_weight=0*self.weight.data

  def forward(self, x):
    self.mask=0*self.weight.data
    self.topk_indices=torch.topk(self.weight, self.keep_in_features, 1, largest=True)[1]
    self.mask[torch.arange(self.weight.shape[0])[:,None], self.topk_indices] = 1  ##  Change those values to 1
    self.net_weight=self.mask*self.weight
    w_times_x= torch.mm(x, self.net_weight.t())
    return w_times_x


In [10]:
from math import floor
#@title Define layer class that uses a block structure
class BlockLinear(nn.Linear):
  """ Custom Linear layer that only uses the stongest K weights """
  def __init__(self, in_features, out_features, bias=False):
    super().__init__(in_features, out_features, bias=bias)
    block_size=floor(in_features/out_features)
    assert(block_size*out_features==in_features)
    self.block_size = block_size

    self.mask=torch.zeros(out_features, in_features, device='cuda:0')
    for counter in range(self.out_features):
      self.mask[counter, torch.arange(counter*self.block_size, (1+counter)*self.block_size)]=1

  def forward(self, x):
    self.net_weight=self.weight*self.mask
    w_times_x= torch.mm(x, self.net_weight.t())
    return w_times_x


In [11]:

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


Using device: cpu
Output: tensor([[ 1.9697,  0.8394, -0.8419]])


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


Using device: cpu
Output: tensor([[-2.2156,  0.1361,  0.5228]])


In [13]:


# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


Output: tensor([[0.6153, 1.3370, 0.5396]])


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


Using device: cpu
Output: tensor([[0.4888, 0.1954, 0.3186]])


In [15]:
#@title Define loss-function & optimizer

# Initialize the optimizer with the model's parameters
lr = 0.01
optimizer = torch.optim.Adam(net.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

print("Model and optimizer initialized successfully.")


lr=0.01
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam( net.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
lossi=[]

print(lr)

ValueError: optimizer got an empty parameter list

In [ ]:
#@title Training the model

for epoch in range(3*num_epochs):
  for i ,(images_raw,labels_raw) in enumerate(train_gen):
    images = transform_images(images_raw.view(-1,28*28)).cuda()
    labels = sum_labels(labels_raw).cuda()

    optimizer.zero_grad()
    outputs = net(inputs) = net(images)
    loss = loss_function(outputs = net(inputs), labels)
    loss.backward()
    optimizer.step()
    net.trimWeights()
    lossi.append(loss.item())

    if (i+1) % 100 == 0:
      scheduler.step()
      print('Epoch [%d/%d], Step [%d/%d], Loss: %.4f'
                 %(epoch+1, num_epochs, i+1, len(train_data)//batch_size, loss.data))

In [ ]:
#@title Evaluating the accuracy of the model

correct = 0
total = 0
for images_raw,labels_raw in test_gen:
  images = transform_images(images_raw.view(-1,28*28)).cuda()
  labels = sum_labels(labels_raw).cuda()

  with torch.no_grad():
    output = net(images)
    _, predicted = torch.max(output,1)
    correct += (predicted == labels).sum()
    total += labels.size(0)

print('Accuracy of the model: %.3f %%' %((100*correct)/(total+1)))

In [ ]:
]# Proportion of labels in training data
processed_labels = np.empty((1,))
for images_raw,labels_raw in train_gen:
  labels = sum_labels(labels_raw).cuda()
  processed_labels = np.concatenate((processed_labels, labels.cpu().numpy()))

processed_labels = processed_labels[1:]


In [ ]:
np.unique(processed_labels, return_counts=True)

In [ ]:
plt.hist(processed_labels)

In [ ]:
plt.imshow(images[1].cpu().numpy().reshape(56,28))

In [ ]:
plt.plot(lossi)
scheduler.get_last_lr()

In [ ]:
net

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
net

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
# plotWithColor(net.dendrites_to_soma.mask.data.cpu())

In [ ]:
labels_sort, labels_sort_ind=torch.sort(labels)

images_sort=images[labels_sort_ind]
plt.plot(labels_sort.cpu().numpy())

images_solo=images_raw.view(-1,28*28).cuda()
labels_solo_sort, labels_solo_sort_ind=torch.sort(labels_raw)
images_solo_sort=images_solo[labels_solo_sort_ind]
plt.plot(labels_solo_sort.cpu().numpy())



In [ ]:
with torch.no_grad():
  output=net(transform_images(images_solo_sort))


In [ ]:
net

In [ ]:
plt.rcParams["figure.figsize"] = (20,3)
plotWithColor(net.inhibition_act.t(), 'inhibition_act')
plotWithColor(net.inhibition_out.t(), 'inhibition_out')

In [ ]:
plotWithColor(net.dendrite_branch_inhibition, 'dendrite_branch_inhibition')
plotWithColor(net.dendrite_branch_excitation, 'dendrite_branch_excitation')
plotWithColor(net.dendrite_branch_act, 'dendrite_branch_act')
plotWithColor(net.dendrite_branch_out, 'dendrite_branch_out')


In [ ]:
plotWithColor(net.dendrite_inhibition.t(), 'dendrite_inhibition')
plotWithColor(net.dendrite_excitation.t(), 'dendrite_excitation')
plotWithColor(net.dendrite_depolarization.t(), 'dendrite_depolarization')
plotWithColor(net.dendrite_act.t(), 'dendrite_act')
plotWithColor(net.dendrite_out.t(), 'dendrite_out')

In [ ]:
plt.rcParams["figure.figsize"] = (10,3)
plotWithColor(net.soma_act.t(), 'soma_act')
# plotWithColor(net.soma_out.t())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

# Check if MPS is available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define your model architecture here
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define layers here, e.g.,
        # self.layer1 = nn.Linear(input_size, hidden_size)
        # self.layer2 = nn.Linear(hidden_size, output_size)
        pass

    def forward(self, x):
        # Define the forward pass
        # Example:
        # x = self.layer1(x)
        # x = nn.ReLU()(x)
        # x = self.layer2(x)
        return x

net = Net().to(device)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

net.apply(init_weights)

# Example input data
inputs = torch.randn(1, 3).to(device)
outputs = net(inputs)
print("Output:", outputs)


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# net =torch.save(net.state_dict(), '/content/gdrive/My Drive/Colab Models/cond_multi_MNIST_mod_sum')

In [ ]:
!ls